# Creating the complete San Francisco film location dataset

The dataset is built with the combination of two sources:
- The Film Location database of the open data portal of San Francisco (OpenSF): https://data.sfgov.org/Culture-and-Recreation/Film-Locations-in-San-Francisco/yitu-d5am/about_data
- OMDb API for obtaining movie and series information: https://www.omdbapi.com/

### Libraries to use

In [1]:
from dotenv import load_dotenv
import os
import requests
import time
import json
import pandas as pd

### Film Location database (OpenSF)
- It originally contains 2214 rows and 18 columns
- There are 350 unique titles

In [ ]:
# Read the csv file and show the columns and the first rows to quickly explore the data
df = pd.read_csv("data/Film_Locations_in_San_Francisco_20260414.csv")
# print(df.columns)
# print(df.head(3))

In [3]:
# Remove the columns that are not needed for the analysis
df = df.drop(columns=["Point", "Supervisor District", "data_as_of", "data_loaded_at"])

# Remove leading and trailing spaces from the Title column
df["Title"] = df["Title"].str.strip()

The following little code was used to save the unique titles in a csv file
- The Release Year was included to ensure the correct title was retrieved
- With that file, all the 350 unique titles were manually searched to mark them in case they had a different name in IMDb (OMDb)
- If they had a different name or were specific episodes of a series, the IMDb ID was added to them (manual)
- If they were not found in IMDb, they were flagged as "delete"

In [ ]:
# Save the unique titles with their year in a csv file

# df_unique_titles = df[["Title", "Release Year"]].drop_duplicates()
# df_unique_titles.to_csv("unique_titles.csv", index=False)

- Quick Note: "Nash Bridges" and "Super Pumped: The Battle for Uber" have two different Release Years as they are series

### Adapting the unique titles with the IMDB data and format

In [4]:
# Load the IMDB specifications file which was manually made
df_imdb = pd.read_csv("data/unique_titles.csv")

# Merge IMDB specifications into the original dataset
df = df.merge(
    df_imdb[["Title", "IMDB specification"]],   # only bring Title + IMDB columns
    on="Title",
    how="left"                           # keep all rows from original dataset
)

# Fill any titles not in the specs file with empty string
df["IMDB specification"] = df["IMDB specification"].fillna("")

Verification of the merging and information about it

In [ ]:
# Count empty IMDb IDs
empty_count    = (df["IMDB specification"].str.strip() == "").sum()
# Count the "delete" specifications
delete_count   = (df["IMDB specification"].str.strip().str.lower() == "delete").sum()
# Count the valid IMDb IDs (all start with "tt")
tt_count       = df["IMDB specification"].str.strip().str.startswith("tt").sum()

# Catch anything unexpected in the IMDB column
unexpected     = df[
    ~(df["IMDB specification"].str.strip() == "") &
    ~(df["IMDB specification"].str.strip().str.lower() == "delete") &
    ~(df["IMDB specification"].str.strip().str.startswith("tt"))
]["IMDB specification"].unique()

print(f"Search by title:         {empty_count}")
print(f"To delete:               {delete_count}")
print(f"IMDb ID (tt...):         {tt_count}")
print(f"Unexpected values:       {unexpected}")

Search by title:         1020
To delete:               16
IMDb ID (tt...):         1224
Unexpected values:       []


- 16 rows (values) were deleted as they were not in the IMDB base or were only one episode of series which location was not in San Francisco (Ex. CSI: NY - episode 903)
- Those 16 rows corresponded to 7 Unique Titles
- 1224 rows are going to be searched by the manually written IMDB ID, that is for 156 Unique Titles
- 1020 rows are going to be searched by the Title (which needs an exact match), that is for 189 Unique Titles

### Retrieve the data from the OMDb API

In [6]:
# Get an API Key in the following site: https://www.omdbapi.com/apikey.aspx
load_dotenv()
OMDB_API_KEY = os.getenv("OMDB_API_KEY")    # Use of env for security reasons

In [7]:
def fetch_movie_by_id(imdb_id):
    """Fetch movie data from OMDB API by IMDb ID"""
    url = "http://www.omdbapi.com/"
    params = {
        "i":      imdb_id.strip(),
        "apikey": OMDB_API_KEY,
        "plot":   "full"
    }
    response = requests.get(url, params=params, timeout=10)
    return response.json()

def fetch_movie_by_title(title):
    """Fetch movie data from OMDB API by title"""
    url = "http://www.omdbapi.com/"
    params = {
        "t":      title.strip(),
        "apikey": OMDB_API_KEY,
        "plot":   "full"
    }
    response = requests.get(url, params=params, timeout=10)
    return response.json()



# First, remove all rows where the IMDb ID is "delete"
titles_to_delete = df[df["IMDB specification"].str.strip().str.lower() == "delete"]["Title"].unique()

# Remove rows with "delete" specification
if len(titles_to_delete) > 0:
    df = df[~df["Title"].isin(titles_to_delete)].reset_index(drop=True)

print(f"Rows after deletions: {len(df)}")



# Second, build a unique title + IMDb ID lookup
df_unique = (df[["Title", "IMDB specification"]]
             .drop_duplicates(subset=["Title"])
             .reset_index(drop=True))

# There should be 343, as 350 originally minus 7 deleted titles
print(f"Unique titles to fetch: {len(df_unique)}")



# Third, fetch the data
data = []
failed = []

# Iterate over unique titles and their IMDb specifications
for _, row in df_unique.iterrows():
    title = str(row["Title"]).strip()
    imdb_spec = str(row["IMDB specification"]).strip() if pd.notna(row["IMDB specification"]) else ""

    try:
        # Rule: IMDb ID provided → search by ID first
        if imdb_spec.lower().startswith("tt"):
            result = fetch_movie_by_id(imdb_spec)
            search_method = "imdb_id"

        # Rule: Empty spec → search by title
        elif imdb_spec == "" or imdb_spec.lower() == "nan":
            result = fetch_movie_by_title(title)
            search_method = "title"

        # If there is an unexpected value → log and skip
        else:
            failed.append(title)
            continue

        # OMDb returns {"Response": "False"} when nothing is found
        if result.get("Response") == "False":
            failed.append(title)
            continue

        data.append({
            # search metadata
            "title_searched":       title,

            # identity
            "imdb_id":              result.get("imdbID"),
            "title":                result.get("Title"),
            "year":                 result.get("Year"),
            "kind":                 result.get("Type"),

            # content
            "genres":               result.get("Genre"),
            "plot":                 result.get("Plot"),
            "language":             result.get("Language"),
            "country":              result.get("Country"),
            "certificates":         result.get("Rated"),
            "runtime":              result.get("Runtime"),
            "awards":               result.get("Awards"),

            # people
            "directors":            result.get("Director"),
            "writers":              result.get("Writer"),

            # ratings
            "imdb_rating":          result.get("imdbRating"),
            "imdb_votes":           result.get("imdbVotes"),
            "metascore":            result.get("Metascore"),
            "ratings":              result.get("Ratings"),

            # commercial
            "box_office":           result.get("BoxOffice"),

            # media
            "poster":               result.get("Poster"),
        })

    # for errors
    except Exception as e:
        failed.append(title)

    # to avoid rate limits
    finally:
        time.sleep(0.3)

Rows after deletions: 2244
Unique titles to fetch: 343


### Saving the json of the fetched data

In [8]:
temp_path  = "data/omdb_data_temp.json"
final_path = "data/omdb_data.json"

if data:
    # Write to temp file first
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    # Only replace original once temp is fully written
    os.replace(temp_path, final_path)
    print(f"Saved {len(data):,} records to {final_path}")
else:
    print("Nothing saved")

# Summary
print(f"Fetched:  {len(data):,}")
print(f"Failed:   {len(failed):,}")

Saved 343 records to data/omdb_data.json
Fetched:  343
Failed:   0


### Join the fetched data with the original SF dataset

In [11]:
# Load the OMDb json file 
with open("data/omdb_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df_imdb = pd.DataFrame(data)

# Replace "N/A" strings with actual NaN
df_imdb = df_imdb.replace("N/A", pd.NA)

# Convert numeric columns
df_imdb["imdb_rating"] = pd.to_numeric(
    df_imdb["imdb_rating"], errors="coerce"
)
df_imdb["imdb_votes"] = (
    df_imdb["imdb_votes"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Ensure numeric year in common format (4-digit year)
df_imdb["year"] = pd.to_numeric(
    df_imdb["year"].astype(str).str.extract(r'(\d{4})')[0],
    errors="coerce"
)

# Flatten nested ratings column
def extract_rating(ratings_list, source):
    if not isinstance(ratings_list, list):
        return pd.NA
    for r in ratings_list:
        if r.get("Source") == source:
            return r.get("Value")
    return pd.NA

df_imdb["rt_score"]         = df_imdb["ratings"].apply(
    lambda x: extract_rating(x, "Rotten Tomatoes"))
df_imdb["metacritic_score"] = df_imdb["ratings"].apply(
    lambda x: extract_rating(x, "Metacritic"))
df_imdb["imdb_score"]       = df_imdb["ratings"].apply(
    lambda x: extract_rating(x, "Internet Movie Database"))

df_imdb = df_imdb.drop(columns=["ratings"])

# Print basic info
print(f"Total records:         {len(df_imdb):,}")
print(f"Columns:               {len(df_imdb.columns)}")
print(f"Movies (kind=movie):   {(df_imdb['kind'] == 'movie').sum()}")
print(f"Series (kind=series):  {(df_imdb['kind'] == 'series').sum()}")

# Print info about missing values
print("\nMissing values per column:")
missing = df_imdb.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)



# Merge data with the original SF dataset
df_final = df.merge(
    df_imdb,
    left_on="Title",            # original CSV title column
    right_on="title_searched",  # OMDB searched title column
    how="left"                  # keep ALL location rows even if no OMDB match
)

# Check the merge
print(f"\nMerge Summary:")
print(f"Original SF rows:      {len(df):,}")
print(f"After merge rows:      {len(df_final):,}")
print(f"Matched rows:          {df_final['imdb_id'].notna().sum():,}")
print(f"Unmatched rows:        {df_final['imdb_id'].isna().sum():,}")
print(f"Total columns:         {len(df_final.columns)}")

# List the unmatched titles (should be 7, as per the "delete" specifications)
unmatched = df_final[df_final["imdb_id"].isna()]["Title"].unique()
if len(unmatched) > 0:
    print(f"\nUnmatched titles ({len(unmatched)}):")
    for t in sorted(unmatched):
        print(f"{t}")

Total records:         343
Columns:               22
Movies (kind=movie):   291
Series (kind=series):  52

Missing values per column:
metacritic_score    144
metascore           144
box_office          141
rt_score             95
awards               65
directors            52
certificates         21
runtime               9
imdb_score            5
writers               5
imdb_rating           5
imdb_votes            4
poster                3
language              1
dtype: int64

Merge Summary:
Original SF rows:      2,244
After merge rows:      2,244
Matched rows:          2,244
Unmatched rows:        0
Total columns:         37


Clean the combined dataset for the final result

In [12]:
# Fill the empty "Writer" values of the original dataset with the "writers" from OMDB, if available
df_final["Writer"] = df_final["Writer"].fillna(df_final["writers"])

# Clean "runtime" column and convert to numeric minutes
def clean_runtime(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    value = value.lower().replace("min", "").strip()
    value = ''.join(c for c in value if c.isdigit() or c == '.')
    try:
        return float(value) if '.' in value else int(value)
    except ValueError:
        return pd.NA

df_final["runtime"] = df_final["runtime"].apply(clean_runtime)

# Drop the unnedded columns
columns_to_drop = [
    "Title",
    "Release Year",
    "directors",
    "writers",
    "title_searched",
    "IMDB specification",
    "imdb_score"
]

existing_to_drop = [c for c in columns_to_drop if c in df_final.columns]
missing_to_drop  = [c for c in columns_to_drop if c not in df_final.columns]

if missing_to_drop:
    print(f"\nColumns not found and skipped:")
    for c in missing_to_drop:
        print(f"{c}")

df_final = df_final.drop(columns=existing_to_drop)

# Check shape after dropping
print(f"Shape after dropping:   {df_final.shape}")


# Delete rows for unmatched titles
rows_to_remove = df_final[df_final["imdb_id"].isna()].shape[0]
df_final = df_final[df_final["imdb_id"].notna()].reset_index(drop=True)
print(f"Rows removed:          {rows_to_remove}")
print(f"Final shape:           {df_final.shape}")


# Capitalize all column names
df_final.columns = [col.capitalize() for col in df_final.columns]

# Fix specific column names after capitalizing
df_final = df_final.rename(columns={
    "Imdb_id":          "IMDB_id",
    "Imdb_rating":      "IMDB_rating",
    "Imdb_votes":       "IMDB_votes",
    "Rt_score":         "RT_score",
})



# Move Title column to the front
cols = ["Title"] + [col for col in df_final.columns if col != "Title"]
df_final = df_final[cols]



# Save cleaned dataset to a new CSV file
temp_path  = "data/sf_movies_cleaned_temp.csv"
final_path = "data/sf_movies_cleaned.csv"

df_final.to_csv(temp_path, index=False, encoding="utf-8")
os.replace(temp_path, final_path)

print(f"\nSaved cleaned dataset to '{final_path}'")
print(f"  Shape:     {df_final.shape}")
print(f"  File size: {os.path.getsize(final_path) / 1e6:.2f} MB")

Shape after dropping:   (2244, 30)
Rows removed:          0
Final shape:           (2244, 30)

Saved cleaned dataset to 'data/sf_movies_cleaned.csv'
  Shape:     (2244, 30)
  File size: 2.27 MB


### Important notes
- The Box Office column is not included in all movies, and it also shows only the "Domestic" office, that is US and Canada as stated in [IMDb help site](https://help.imdb.com/article/imdbpro/industry-research/box-office-mojo-by-imdbpro-faq/GCWTV4MQKGWRAUAP#)
- There are 202 movies with Box Office information, while 114 do not have that information